# LLM Function Calling & Tool-Using Agents
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/06_GenAI_LLM_RAG/llm_function_calling_agents.ipynb)

Agents = LLM + tools + loop. The model sees a list of callable functions, decides which to invoke with which arguments, observes the result and repeats until it can answer.

This notebook builds a **minimal ReAct-style agent loop in plain Python**. The `llm_decide()` slot is mocked deterministically so everything runs offline/free - swap in any real model (OpenAI tools API, Ollama, HF) at the marked line.

## 1. Define tools

In [ ]:
import json

TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression",
        "parameters": {"expression": "string"},
        "run": lambda expression: str(eval(expression, {"__builtins__": {}}, {})),
    },
    "get_weather": {
        "description": "Current weather for a city",
        "parameters": {"city": "string"},
        "run": lambda city: {"Pune": "29C sunny", "Mumbai": "31C humid",
                             "Delhi": "24C haze"}.get(city.title(), "18C cloudy"),
    },
}
print(json.dumps({k: v["description"] for k, v in TOOLS.items()}, indent=2))

### What providers send the model (reference)

```json
{"name": "get_weather",
 "arguments": {"city": "Pune"}}
```

OpenAI calls it *tools*, Anthropic *tool use*, Gemini *function calling* - same idea: the model emits JSON, YOUR code executes it and feeds the output back.

## 2. The agent loop (ReAct: reason -> act -> observe)

In [ ]:
def llm_decide(question, history):
    """MOCK model call. Replace with a real LLM that reads TOOLS + history
    and returns {'tool': name, 'arguments': {...}} or {'final': str}."""
    last = history[-1]["content"] if history else question
    low = last.lower()
    if any(op in low for op in ["+", "-", "*", "/"]) and "calculate" not in low:
        expr = "".join(ch for ch in low.split("calculate")[-1] if ch.isdigit() or ch in "+-*/(). ")
        return {"tool": "calculator", "arguments": {"expression": expr.strip()}}
    for city in ["pune", "mumbai", "delhi"]:
        if city in low:
            return {"tool": "get_weather", "arguments": {"city": city}}
    if history:
        obs = history[-1]
        return {"final": f"Based on the tool result: {obs['content']}"}
    return {"final": "I need more information."}

def run_agent(question, max_steps=5):
    history = [{"role": "user", "content": question}]
    for step in range(max_steps):
        decision = llm_decide(question, history)
        if "final" in decision:
            print(f"[step {step}] FINAL -> {decision['final']}")
            return decision["final"]
        tool, args = decision["tool"], decision["arguments"]
        observation = TOOLS[tool]["run"](**args)
        print(f"[step {step}] CALL {tool}({args}) -> {observation}")
        history.append({"role": "tool", "content": f"{tool}: {observation}"})
    return "max steps reached"

print("Q1:"); run_agent("What's 128 * 46 plus 19?")
print("\nQ2:"); run_agent("Should I carry sunglasses in Mumbai today?")

## 3. Making it real (swap-in guide)

```python
# Ollama (free, local):  see ollama_local_llm.ipynb
resp = requests.post("http://localhost:11434/api/chat", json={
    "model": "llama3.2",
    "messages": system_prompt_with_tools + history,
    "stream": False})
decision = json.loads(resp.json()["message"]["content"])
```

Production notes:
- Validate arguments with pydantic before executing anything.
- Sand-box dangerous tools (SQL, shell, payments) - treat model output as untrusted input.
- Cap iterations; log every step for debugging.